In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
df = spark.table("databriks.bronze.order_items")
display(df)

In [0]:
(df.count() , len(df.columns))

In [0]:
df.printSchema()

In [0]:
df = df.withColumnRenamed("dt", "Date").withColumnRenamed("order_ts","order_timestamp")
df = df.withColumn("coupon_code", F.when(F.col("coupon_code").isNull(), "No Coupon").otherwise(F.col("coupon_code")))
df = df.withColumn("channel", F.upper(F.col("channel")))
df = df.withColumn("order_timestamp", F.to_timestamp(F.col("order_timestamp")))
display(df.groupBy("order_id").count().where(F.col("count") > 1))

In [0]:
df = df.dropDuplicates(["order_id", "item_seq"])
display(df.groupBy(["order_id", "item_seq"]).count().where(F.col("count") > 1))

In [0]:
df = df.withColumn("quantity", F.when(F.col("quantity").isNull(), 0).when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int"))
df = df.withColumn("discount_pct" , F.regexp_replace(F.col("discount_pct"), "%", ""))
display(df)


In [0]:
df = df.withColumn(
    "unit_price",
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

In [0]:
df = df.withColumn("discount_pct", F.col("discount_pct").cast("double"))
df = df.withColumn("tax_amount", F.col("tax_amount").cast("double"))
df.printSchema()

In [0]:
df.write.format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable("databriks.silver.Orders")